# Convert Final 8-Sample Seed123 H5 To ONNX

This notebook converts the final deployment H5 model for `seed=123` into ONNX.

Target model:
- `training_outputs/mobilenetv3small_8samples_final_deployment_cnn_only/models/meatlens_final_8samples_cnn_only_mobilenetv3small_seed123.h5`

Output:
- `training_outputs/mobilenetv3small_8samples_final_deployment_cnn_only/models/meatlens_final_8samples_cnn_only_mobilenetv3small_seed123.onnx`


In [1]:
from pathlib import Path
import json
import importlib

PROJECT_ROOT = Path.cwd()
MODELS_ROOT = PROJECT_ROOT / "training_outputs" / "mobilenetv3small_8samples_final_deployment_cnn_only" / "models"

MODEL_H5_PATH = MODELS_ROOT / "meatlens_final_8samples_cnn_only_mobilenetv3small_seed123.h5"
MODEL_ONNX_PATH = MODELS_ROOT / "meatlens_final_8samples_cnn_only_mobilenetv3small_seed123.onnx"
MODEL_METADATA_PATH = MODELS_ROOT / "meatlens_final_8samples_cnn_only_mobilenetv3small_seed123_metadata.json"

print("MODEL_H5_PATH =", MODEL_H5_PATH)
print("MODEL_ONNX_PATH =", MODEL_ONNX_PATH)
print("MODEL_METADATA_PATH =", MODEL_METADATA_PATH)
print("H5 exists?", MODEL_H5_PATH.exists())
print("Metadata exists?", MODEL_METADATA_PATH.exists())


MODEL_H5_PATH = e:\Thesis Code\training_outputs\mobilenetv3small_8samples_final_deployment_cnn_only\models\meatlens_final_8samples_cnn_only_mobilenetv3small_seed123.h5
MODEL_ONNX_PATH = e:\Thesis Code\training_outputs\mobilenetv3small_8samples_final_deployment_cnn_only\models\meatlens_final_8samples_cnn_only_mobilenetv3small_seed123.onnx
MODEL_METADATA_PATH = e:\Thesis Code\training_outputs\mobilenetv3small_8samples_final_deployment_cnn_only\models\meatlens_final_8samples_cnn_only_mobilenetv3small_seed123_metadata.json
H5 exists? True
Metadata exists? True


In [2]:
import importlib.util

def package_available(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

print("tensorflow available?", package_available("tensorflow"))
print("tf2onnx available?", package_available("tf2onnx"))
print("onnx available?", package_available("onnx"))


tensorflow available? True
tf2onnx available? True
onnx available? True


In [3]:
import tensorflow as tf
import tf2onnx

try:
    import onnx
except Exception:
    onnx = None

def convert_seed123_h5_to_onnx(
    model_h5_path=MODEL_H5_PATH,
    model_onnx_path=MODEL_ONNX_PATH,
    opset=13,
):
    if not model_h5_path.exists():
        raise FileNotFoundError(f"Missing H5 model: {model_h5_path}")

    model = tf.keras.models.load_model(model_h5_path, compile=False)

    input_signature = (
        tf.TensorSpec((None, 224, 224, 3), tf.float32, name="image_input"),
    )

    model_proto, external_tensor_storage = tf2onnx.convert.from_keras(
        model,
        input_signature=input_signature,
        opset=opset,
        output_path=str(model_onnx_path),
    )

    result = {
        "model_h5_path": str(model_h5_path),
        "model_onnx_path": str(model_onnx_path),
        "opset": opset,
        "onnx_exists": model_onnx_path.exists(),
        "external_tensor_storage": external_tensor_storage,
    }

    if model_onnx_path.exists():
        result["onnx_size_mb"] = model_onnx_path.stat().st_size / (1024 * 1024)

    if onnx is not None and model_onnx_path.exists():
        onnx_model = onnx.load(str(model_onnx_path))
        onnx.checker.check_model(onnx_model)
        result["onnx_inputs"] = [node.name for node in onnx_model.graph.input]
        result["onnx_outputs"] = [node.name for node in onnx_model.graph.output]

    return result


In [4]:
if MODEL_METADATA_PATH.exists():
    metadata = json.loads(MODEL_METADATA_PATH.read_text(encoding="utf-8"))
    metadata
else:
    print("Metadata JSON not found.")


In [6]:
MANUAL_CONFIRM_CONVERT_TO_ONNX = True

if MANUAL_CONFIRM_CONVERT_TO_ONNX:
    onnx_result = convert_seed123_h5_to_onnx()
    onnx_result
else:
    print("Seed123 H5 to ONNX conversion is ready but not started.")
    print("Set MANUAL_CONFIRM_CONVERT_TO_ONNX = True to convert.")


After conversion, the ONNX file should be here:

- `training_outputs/mobilenetv3small_8samples_final_deployment_cnn_only/models/meatlens_final_8samples_cnn_only_mobilenetv3small_seed123.onnx`
